# Fase 07: Carga en base de datos SQLite

## Proyecto II Parcial - Modelado Avanzado de Base de Datos

Este notebook implementa la **Fase 07: Carga en base de datos**.

Objetivo:

- Tomar las tablas finales generadas en las fases anteriores.
- Crear una base de datos SQLite local.
- Cargar las tablas obligatorias del proyecto.
- Ejecutar consultas SQL para demostrar que la carga fue exitosa.

Base usada: **SQLite**.

SQLite no necesita servidor, usuario, contraseña ni puerto. Python ya incluye el módulo `sqlite3` por defecto.


## 1. Tablas obligatorias que se cargarán

La Fase 07 exige cargar estas tablas en una base de datos consultable:

1. `gold_trips_clean`
2. `gold_daily_revenue`
3. `gold_location_performance`
4. `quality_rejected_records`
5. `quality_metrics_summary`
6. `audit_file_inventory`

Este notebook busca automáticamente la versión más reciente de cada tabla en las carpetas:

- `data/gold/`
- `data/quarantine/`
- `data/audit/`


In [1]:
# Preparar Python, Hadoop y Spark para trabajar en Windows.

import os
import sys
import sqlite3
import json
import math
import decimal
from pathlib import Path
from datetime import datetime, date

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Ajusta estas rutas solo si en tu computadora Hadoop o Java están en otra ubicación.
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")

# Si no tienes JAVA_HOME configurado, descomenta y ajusta esta línea:
# os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"

print("Python usado por PySpark:", sys.executable)
print("SQLite disponible correctamente.")

Python usado por PySpark: c:\Users\Usuario\AppData\Local\Programs\Python\Python311\python.exe
SQLite disponible correctamente.


In [2]:
# Importar Spark.

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    FloatType,
    BooleanType,
    TimestampType,
    DateType,
    DecimalType,
    ShortType,
    ByteType
)

In [3]:
# Crear sesión Spark.

spark = (
    SparkSession.builder
    .appName("Fase_07_Carga_SQLite")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.files.ignoreCorruptFiles", "true")
    .config("spark.sql.files.ignoreMissingFiles", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark listo.")
print("Versión Spark:", spark.version)

Spark listo.
Versión Spark: 4.1.2


In [4]:
# Rutas principales del proyecto.
# Si ejecutas el notebook desde la carpeta notebooks, se toma la carpeta padre como raíz.

BASE_PATH = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

DATA_PATH = BASE_PATH / "data"
GOLD_PATH = DATA_PATH / "gold"
AUDIT_PATH = DATA_PATH / "audit"
QUARANTINE_PATH = DATA_PATH / "quarantine"
DATABASE_PATH = DATA_PATH / "database"

DATABASE_PATH.mkdir(parents=True, exist_ok=True)

DB_FILE = DATABASE_PATH / "etl_taxi_gold.db"
PROCESS_ID = "fase7_" + datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("AUDIT_PATH:", AUDIT_PATH)
print("QUARANTINE_PATH:", QUARANTINE_PATH)
print("DATABASE_PATH:", DATABASE_PATH)
print("DB_FILE:", DB_FILE)
print("PROCESS_ID:", PROCESS_ID)

BASE_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced
GOLD_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\gold
AUDIT_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\audit
QUARANTINE_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine
DATABASE_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\database
DB_FILE: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\database\etl_taxi_gold.db
PROCESS_ID: fase7_20260617_183040


In [5]:
# Función auxiliar para convertir rutas Windows a formato aceptado por Spark.

def spark_path(path):
    return str(Path(path).resolve()).replace("\\", "/")

print("Ejemplo de ruta Spark:")
print(spark_path(DB_FILE))

Ejemplo de ruta Spark:
C:/Users/Usuario/Downloads/etl_spark_parquet_advanced/etl_spark_parquet_advanced/data/database/etl_taxi_gold.db


## 2. Buscar las tablas Parquet más recientes

Las fases anteriores guardan las tablas en carpetas con `process_id`, por ejemplo:

- `gold_trips_clean_fase6_...`
- `quality_metrics_summary_fase5_...`
- `audit_file_inventory_fase1_...`

Por eso esta función busca la carpeta válida más reciente que contenga archivos `.parquet` reales.


In [6]:
# Funciones para encontrar carpetas Parquet válidas.

def list_real_parquet_files(path):
    """Devuelve archivos parquet reales, ignorando temporales y metadatos."""
    path = Path(path)
    if not path.exists():
        return []

    return [
        file for file in path.rglob("*.parquet")
        if "_temporary" not in str(file).lower()
        and not file.name.startswith(".")
        and file.is_file()
    ]


def find_latest_parquet_folder(base_path, table_name):
    """Busca la carpeta más reciente que contenga el nombre de la tabla y archivos parquet."""
    base_path = Path(base_path)
    candidates = []

    if not base_path.exists():
        return None

    # Búsqueda directa en carpetas cuyo nombre contiene la tabla.
    for path in base_path.glob(f"*{table_name}*"):
        if path.is_dir():
            parquet_files = list_real_parquet_files(path)
            if len(parquet_files) > 0:
                latest_file_time = max(file.stat().st_mtime for file in parquet_files)
                candidates.append({
                    "path": path,
                    "modified_time": latest_file_time,
                    "parquet_count": len(parquet_files)
                })

    # Búsqueda recursiva por si la carpeta está más adentro.
    if len(candidates) == 0:
        for path in base_path.rglob(f"*{table_name}*"):
            if path.is_dir():
                parquet_files = list_real_parquet_files(path)
                if len(parquet_files) > 0:
                    latest_file_time = max(file.stat().st_mtime for file in parquet_files)
                    candidates.append({
                        "path": path,
                        "modified_time": latest_file_time,
                        "parquet_count": len(parquet_files)
                    })

    if len(candidates) == 0:
        return None

    latest = max(candidates, key=lambda item: item["modified_time"])
    return latest


print("Funciones de búsqueda listas.")

Funciones de búsqueda listas.


In [7]:
# Definir las tablas obligatorias y la carpeta donde deberían estar.

tables_to_load = {
    "gold_trips_clean": GOLD_PATH,
    "gold_daily_revenue": GOLD_PATH,
    "gold_location_performance": GOLD_PATH,
    "quality_rejected_records": QUARANTINE_PATH,
    "quality_metrics_summary": AUDIT_PATH,
    "audit_file_inventory": AUDIT_PATH
}

discovered_tables = {}
missing_tables = []

for table_name, base_path in tables_to_load.items():
    latest_info = find_latest_parquet_folder(base_path, table_name)

    if latest_info is None:
        print(f"NO SE ENCONTRÓ: {table_name}")
        missing_tables.append(table_name)
    else:
        discovered_tables[table_name] = latest_info["path"]
        print("=" * 80)
        print(f"Tabla encontrada: {table_name}")
        print("Ruta:", latest_info["path"])
        print("Archivos parquet:", latest_info["parquet_count"])

print("=" * 80)
print("Tablas encontradas:", list(discovered_tables.keys()))
print("Tablas faltantes:", missing_tables)

Tabla encontrada: gold_trips_clean
Ruta: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\gold\gold_trips_clean_fase6_20260617_002625
Archivos parquet: 11
Tabla encontrada: gold_daily_revenue
Ruta: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\gold\gold_daily_revenue_fase6_20260617_002625
Archivos parquet: 3
Tabla encontrada: gold_location_performance
Ruta: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\gold\gold_location_performance_fase6_20260617_002625
Archivos parquet: 3
Tabla encontrada: quality_rejected_records
Ruta: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine\quality_rejected_records_fase5_20260617_181600
Archivos parquet: 4
Tabla encontrada: quality_metrics_summary
Ruta: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\audit\quality_metrics_summary_fase5_20260617_181600
Archi

## 3. Leer las tablas con Spark

Si alguna tabla aparece como faltante, primero debes ejecutar la fase que la genera:

- Gold: ejecutar Fase 06.
- Calidad: ejecutar Fase 05.
- Auditoría: ejecutar Fase 01.


In [8]:
# Leer cada tabla encontrada con Spark.

spark_tables = {}
row_counts = []

for table_name, table_path in discovered_tables.items():
    print("=" * 80)
    print(f"Leyendo tabla: {table_name}")
    print(table_path)

    df = spark.read.parquet(spark_path(table_path))
    count_rows = df.count()

    spark_tables[table_name] = df
    row_counts.append((table_name, count_rows, str(table_path)))

    print("Registros:", count_rows)
    print("Columnas:", len(df.columns))
    df.printSchema()

print("=" * 80)
print("Resumen de tablas leídas:")
for table_name, count_rows, table_path in row_counts:
    print(f"{table_name}: {count_rows} registros | {table_path}")

Leyendo tabla: gold_trips_clean
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\gold\gold_trips_clean_fase6_20260617_002625
Registros: 43890529
Columnas: 17
root
 |-- trip_id: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tip_percentage: double (nullable = true)
 |-- average_speed_mph: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

Leyendo tabla: gold_daily_r

In [9]:
# Validar que estén todas las tablas obligatorias.

required_tables = list(tables_to_load.keys())
loaded_tables = list(spark_tables.keys())
not_loaded = [table for table in required_tables if table not in loaded_tables]

if len(not_loaded) > 0:
    raise FileNotFoundError(
        "Faltan tablas obligatorias para la Fase 07: " + ", ".join(not_loaded) +
        ". Ejecuta primero las fases anteriores que generan esas tablas."
    )
else:
    print("Validación correcta: todas las tablas obligatorias fueron encontradas y leídas.")

Validación correcta: todas las tablas obligatorias fueron encontradas y leídas.


## 4. Crear funciones para cargar DataFrames Spark en SQLite

Se evita convertir todo el dataset a Pandas para no romper la restricción del proyecto.  
La carga se hace leyendo filas desde Spark y enviándolas por lotes a SQLite.


In [10]:
# Funciones para crear tablas SQLite y cargar datos por lotes.

def quote_identifier(name):
    """Escapa nombres de columnas o tablas para SQLite."""
    return '"' + str(name).replace('"', '""') + '"'


def spark_type_to_sqlite(data_type):
    """Convierte tipos de Spark a tipos simples de SQLite."""
    if isinstance(data_type, (IntegerType, LongType, ShortType, ByteType)):
        return "INTEGER"
    if isinstance(data_type, (DoubleType, FloatType, DecimalType)):
        return "REAL"
    if isinstance(data_type, BooleanType):
        return "INTEGER"
    if isinstance(data_type, (TimestampType, DateType)):
        return "TEXT"
    return "TEXT"


def normalize_value(value):
    """Convierte valores de Spark/Python a valores aceptables para SQLite."""
    if value is None:
        return None

    if isinstance(value, bool):
        return 1 if value else 0

    if isinstance(value, (datetime, date)):
        return value.isoformat(sep=" ") if isinstance(value, datetime) else value.isoformat()

    if isinstance(value, decimal.Decimal):
        return float(value)

    if isinstance(value, float):
        if math.isnan(value) or math.isinf(value):
            return None
        return value

    if isinstance(value, (list, tuple, dict)):
        return json.dumps(value, ensure_ascii=False)

    return value


def create_sqlite_table(conn, table_name, df):
    """Crea una tabla SQLite con base en el esquema del DataFrame Spark."""
    columns_sql = []

    for field in df.schema.fields:
        column_name = quote_identifier(field.name)
        column_type = spark_type_to_sqlite(field.dataType)
        columns_sql.append(f"{column_name} {column_type}")

    create_sql = f"CREATE TABLE {quote_identifier(table_name)} (\n    " + ",\n    ".join(columns_sql) + "\n);"

    conn.execute(f"DROP TABLE IF EXISTS {quote_identifier(table_name)};")
    conn.execute(create_sql)


def load_spark_df_to_sqlite(conn, table_name, df, batch_size=5000):
    """Carga un DataFrame Spark a SQLite por lotes, sin usar Pandas como motor de procesamiento."""
    create_sqlite_table(conn, table_name, df)

    columns = df.columns
    quoted_columns = ", ".join(quote_identifier(col) for col in columns)
    placeholders = ", ".join(["?"] * len(columns))

    insert_sql = f"INSERT INTO {quote_identifier(table_name)} ({quoted_columns}) VALUES ({placeholders});"

    batch = []
    inserted_rows = 0

    for row in df.toLocalIterator():
        row_dict = row.asDict(recursive=False)
        values = tuple(normalize_value(row_dict.get(col)) for col in columns)
        batch.append(values)

        if len(batch) >= batch_size:
            conn.executemany(insert_sql, batch)
            inserted_rows += len(batch)
            batch = []
            print(f"{table_name}: {inserted_rows} filas insertadas...")

    if len(batch) > 0:
        conn.executemany(insert_sql, batch)
        inserted_rows += len(batch)

    conn.commit()
    print(f"{table_name}: carga terminada con {inserted_rows} filas.")
    return inserted_rows


print("Funciones SQLite listas.")

Funciones SQLite listas.


## 5. Cargar las tablas en SQLite

La base se guardará como archivo local en:

`data/database/etl_taxi_gold.db`

Si ejecutas nuevamente el notebook, las tablas se reemplazan. Esto evita duplicados y ayuda a mantener el proceso idempotente.


In [11]:
# Crear base SQLite y cargar tablas obligatorias.

conn = sqlite3.connect(DB_FILE)

# Ajustes para mejorar velocidad de escritura local.
conn.execute("PRAGMA journal_mode = WAL;")
conn.execute("PRAGMA synchronous = NORMAL;")
conn.execute("PRAGMA temp_store = MEMORY;")
conn.execute("PRAGMA cache_size = -200000;")

load_results = []

# Orden recomendado: tablas pequeñas primero, luego gold_trips_clean si es grande.
load_order = [
    "audit_file_inventory",
    "quality_metrics_summary",
    "quality_rejected_records",
    "gold_daily_revenue",
    "gold_location_performance",
    "gold_trips_clean"
]

for table_name in load_order:
    print("=" * 80)
    print(f"Cargando tabla en SQLite: {table_name}")

    inserted_rows = load_spark_df_to_sqlite(
        conn=conn,
        table_name=table_name,
        df=spark_tables[table_name],
        batch_size=5000
    )

    load_results.append({
        "process_id": PROCESS_ID,
        "table_name": table_name,
        "inserted_rows": inserted_rows,
        "loaded_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })

print("=" * 80)
print("Carga de tablas finalizada correctamente.")

Cargando tabla en SQLite: audit_file_inventory
audit_file_inventory: carga terminada con 16 filas.
Cargando tabla en SQLite: quality_metrics_summary
quality_metrics_summary: carga terminada con 11 filas.
Cargando tabla en SQLite: quality_rejected_records
quality_rejected_records: carga terminada con 631 filas.
Cargando tabla en SQLite: gold_daily_revenue
gold_daily_revenue: carga terminada con 331 filas.
Cargando tabla en SQLite: gold_location_performance
gold_location_performance: 5000 filas insertadas...
gold_location_performance: 10000 filas insertadas...
gold_location_performance: 15000 filas insertadas...
gold_location_performance: 20000 filas insertadas...
gold_location_performance: 25000 filas insertadas...
gold_location_performance: 30000 filas insertadas...
gold_location_performance: 35000 filas insertadas...
gold_location_performance: 40000 filas insertadas...
gold_location_performance: 45000 filas insertadas...
gold_location_performance: 50000 filas insertadas...
gold_locati

In [12]:
# Crear una tabla de auditoría de la carga en la misma base SQLite.

conn.execute("DROP TABLE IF EXISTS audit_database_load;")
conn.execute("""
CREATE TABLE audit_database_load (
    process_id TEXT,
    table_name TEXT,
    inserted_rows INTEGER,
    loaded_at TEXT
);
""")

conn.executemany(
    "INSERT INTO audit_database_load (process_id, table_name, inserted_rows, loaded_at) VALUES (?, ?, ?, ?);",
    [
        (
            item["process_id"],
            item["table_name"],
            item["inserted_rows"],
            item["loaded_at"]
        )
        for item in load_results
    ]
)

conn.commit()

print("Tabla audit_database_load creada correctamente.")
for row in conn.execute("SELECT * FROM audit_database_load ORDER BY table_name;"):
    print(row)

Tabla audit_database_load creada correctamente.
('fase7_20260617_183040', 'audit_file_inventory', 16, '2026-06-17 18:30:47')
('fase7_20260617_183040', 'gold_daily_revenue', 331, '2026-06-17 18:30:47')
('fase7_20260617_183040', 'gold_location_performance', 108322, '2026-06-17 18:30:49')
('fase7_20260617_183040', 'gold_trips_clean', 43890529, '2026-06-17 18:58:49')
('fase7_20260617_183040', 'quality_metrics_summary', 11, '2026-06-17 18:30:47')
('fase7_20260617_183040', 'quality_rejected_records', 631, '2026-06-17 18:30:47')


## 6. Verificar tablas creadas en SQLite

Esta sección confirma que las tablas existen dentro de la base de datos y que tienen registros cargados.


In [13]:
# Listar tablas creadas en SQLite.

tables_in_db = conn.execute("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""").fetchall()

print("Tablas existentes en SQLite:")
for table in tables_in_db:
    print("-", table[0])

Tablas existentes en SQLite:
- audit_database_load
- audit_file_inventory
- gold_daily_revenue
- gold_location_performance
- gold_trips_clean
- quality_metrics_summary
- quality_rejected_records


In [14]:
# Contar registros por tabla en SQLite.

print("Conteo de registros en SQLite:")
for table_name in required_tables + ["audit_database_load"]:
    count_sql = f"SELECT COUNT(*) FROM {quote_identifier(table_name)};"
    total = conn.execute(count_sql).fetchone()[0]
    print(f"{table_name}: {total}")

Conteo de registros en SQLite:
gold_trips_clean: 43890529
gold_daily_revenue: 331
gold_location_performance: 108322
quality_rejected_records: 631
quality_metrics_summary: 11
audit_file_inventory: 16
audit_database_load: 6


## 7. Consultas SQL obligatorias de la Fase 07

El enunciado pide demostrar la carga con consultas SQL.  
Las siguientes consultas se ejecutan directamente sobre SQLite.


In [15]:
# Consulta obligatoria 1: ingresos por tipo de servicio.

query_1 = """
SELECT 
    service_type,
    COUNT(*) AS total_trips,
    ROUND(SUM(total_amount), 2) AS total_revenue
FROM gold_trips_clean
GROUP BY service_type
ORDER BY total_revenue DESC;
"""

print("CONSULTA 1: ingresos por servicio")
for row in conn.execute(query_1):
    print(row)

CONSULTA 1: ingresos por servicio
('yellow', 25311957, 587819504.35)
('fhvhv', 18452960, 451103000.09)
('green', 125612, 2736537.93)


In [16]:
# Consulta obligatoria 2: métricas de calidad por servicio, año y mes.

query_2 = """
SELECT 
    service_type,
    year,
    month,
    total_records,
    valid_records,
    rejected_records,
    quality_percentage
FROM quality_metrics_summary
ORDER BY year, month, service_type;
"""

print("CONSULTA 2: métricas de calidad")
for row in conn.execute(query_2):
    print(row)

CONSULTA 2: métricas de calidad
('yellow', 2020, 1, 6281130, 6280948, 182, 100.0)
('yellow', 2021, 1, 1331340, 1331320, 20, 100.0)
('yellow', 2022, 1, 2411789, 2411744, 45, 100.0)
('yellow', 2022, 2, 2920554, 2920496, 58, 100.0)
('fhvhv', 2023, 1, 18452962, 18452960, 2, 100.0)
('green', 2023, 1, 64255, 64252, 3, 100.0)
('yellow', 2023, 1, 2991419, 2991377, 42, 100.0)
('green', 2023, 2, 61381, 61360, 21, 99.97)
('yellow', 2023, 2, 2843569, 2843510, 59, 100.0)
('yellow', 2023, 3, 3319914, 3319805, 109, 100.0)
('yellow', 2023, 4, 3212847, 3212757, 90, 100.0)


In [17]:
# Consulta obligatoria 3: rutas con mayor ingreso.

query_3 = """
SELECT 
    pickup_location_id,
    dropoff_location_id,
    COUNT(*) AS total_trips,
    ROUND(SUM(total_amount), 2) AS total_revenue,
    ROUND(AVG(trip_duration_minutes), 2) AS avg_duration
FROM gold_trips_clean
GROUP BY pickup_location_id, dropoff_location_id
ORDER BY total_revenue DESC
LIMIT 20;
"""

print("CONSULTA 3: rutas con mayor ingreso")
for row in conn.execute(query_3):
    print(row)

CONSULTA 3: rutas con mayor ingreso
(132, 265, 99509, 10553499.0, 46.16)
(138, 265, 44562, 4598039.27, 43.38)
(132, 230, 47241, 3980920.63, 51.47)
(264, 264, 133421, 3195404.82, 14.4)
(138, 230, 49638, 3126321.76, 34.88)
(132, 48, 32090, 2627154.04, 51.64)
(237, 236, 193528, 2593028.91, 6.92)
(230, 138, 38544, 2344687.79, 32.03)
(236, 237, 166680, 2331981.82, 8.09)
(132, 164, 26532, 2205818.31, 42.96)
(230, 132, 25245, 2092167.97, 51.03)
(132, 170, 23011, 1892269.87, 38.69)
(230, 265, 21484, 1838636.25, 36.6)
(138, 161, 28683, 1776816.11, 32.31)
(132, 161, 20705, 1768101.43, 47.53)
(132, 68, 21066, 1757225.6, 48.27)
(161, 265, 19118, 1749652.31, 40.81)
(132, 163, 20523, 1728280.85, 51.69)
(161, 138, 28020, 1687653.48, 30.32)
(132, 162, 19683, 1630499.36, 42.27)


## 8. Consultas adicionales recomendadas

Estas consultas no reemplazan las obligatorias, pero ayudan a defender la Fase 07 en la presentación.


In [18]:
# Consulta adicional 1: validar registros rechazados por regla.

query_extra_1 = """
SELECT
    rejection_rule,
    COUNT(*) AS total_rejections
FROM quality_rejected_records
GROUP BY rejection_rule
ORDER BY total_rejections DESC;
"""

print("Rechazos por regla:")
for row in conn.execute(query_extra_1):
    print(row)

Rechazos por regla:
('PARTITION_DATE_MISMATCH', 614)
('INVALID_AMOUNT_RANGE', 15)
('INVALID_DISTANCE_RANGE', 2)


In [19]:
# Consulta adicional 2: validar inventario de archivos.

query_extra_2 = """
SELECT
    service_type,
    read_status,
    COUNT(*) AS total_files,
    SUM(COALESCE(record_count, 0)) AS total_records
FROM audit_file_inventory
GROUP BY service_type, read_status
ORDER BY service_type, read_status;
"""

print("Inventario de archivos:")
for row in conn.execute(query_extra_2):
    print(row)

Inventario de archivos:
('bad_parquet', 'READ_ERROR', 3, 0)
('bad_parquet', 'READ_OK', 2, 5)
('fhvhv', 'READ_OK', 1, 18479031)
('green', 'READ_OK', 2, 133020)
('yellow', 'READ_OK', 8, 25890876)


In [20]:
# Consulta adicional 3: resumen de carga a base de datos.

query_extra_3 = """
SELECT
    table_name,
    inserted_rows,
    loaded_at
FROM audit_database_load
ORDER BY table_name;
"""

print("Auditoría de carga SQLite:")
for row in conn.execute(query_extra_3):
    print(row)

Auditoría de carga SQLite:
('audit_file_inventory', 16, '2026-06-17 18:30:47')
('gold_daily_revenue', 331, '2026-06-17 18:30:47')
('gold_location_performance', 108322, '2026-06-17 18:30:49')
('gold_trips_clean', 43890529, '2026-06-17 18:58:49')
('quality_metrics_summary', 11, '2026-06-17 18:30:47')
('quality_rejected_records', 631, '2026-06-17 18:30:47')


## 9. Cerrar conexión

Siempre se debe cerrar la conexión a la base de datos al finalizar.


In [21]:
# Cerrar conexión SQLite.

conn.close()

print("Conexión SQLite cerrada correctamente.")
print("Base de datos generada en:")
print(DB_FILE)

Conexión SQLite cerrada correctamente.
Base de datos generada en:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\database\etl_taxi_gold.db




Conclusión de la fase:

> La carga en SQLite permitió demostrar que las tablas finales del Data Lakehouse pueden ser consultadas mediante SQL. Además, se mantuvo trazabilidad de la carga mediante una tabla adicional de auditoría llamada `audit_database_load`, donde se registró el `process_id`, el nombre de cada tabla, la cantidad de registros cargados y la fecha de carga.
